# A/B Testing Tutorial #
This notebook teaches us how to design, simulate, and analyze an A/B test.<br>
<br>
We will use simulated data so that we know the true answer and can check our methods against it.<br>
<br>
The notebook covers:
  - The theory behind A/B tests
  - Comparing two conversion rates with a two-proportion z-test
  - Calculating how many users an experiment needs (sample size)
  - Estimating statistical power by simulation
  - Checking that the test controls false positives (A/A tests)
  - Why peeking at results early inflates false positives
  - Multiple comparisons
<br>

## Theory ##

An **A/B test** is a randomized controlled experiment. Users are split at random into two groups:
  - **Control (A):** sees the current version
  - **Treatment (B):** sees the changed version

Because assignment is random, any reliable difference in the outcome can be attributed to the change.

**Steps in an experiment**
  1. State a hypothesis and choose one primary metric (for example, conversion rate)
  2. Choose a significance level $\alpha$ (usually 0.05) and a desired power (usually 0.8)
  3. Choose the smallest effect worth detecting (the minimum detectable effect, MDE)
  4. Calculate the sample size needed
  5. Randomize users, run the test to the planned end, then analyze

**Hypotheses**

$$
H_0: p_B = p_A \qquad H_1: p_B \neq p_A
$$

**Two-proportion z-test**

With conversions $x_A$ out of $n_A$ users and $x_B$ out of $n_B$ users, the pooled proportion is

$$
\hat{p} = \frac{x_A + x_B}{n_A + n_B}
$$

and the test statistic is

$$
z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}
$$

**Key terms**
  - **p-value:** the probability of a difference at least this large if there were truly no difference
  - **Power:** the probability of detecting a real effect of the size you care about
  - **Type I error:** a false positive, rejecting $H_0$ when it is true. Its rate is $\alpha$.
  - **Type II error:** a false negative, missing a real effect. Its rate is $\beta = 1 - \text{power}$.
<br>

### Simulating an Experiment ###
In a real experiment we never know the true conversion rates. Here we set them ourselves so we can compare our results to the truth.<br>
<br>
We start with a 10% baseline and a true lift of 1.5 percentage points for the treatment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline

rng = np.random.default_rng(42)

In [ ]:
# True conversion rates, unknown in a real experiment
p_control = 0.10
p_treatment = 0.115

# Users needed per group, using the sample size formula explained below
def sample_size_per_group(p_base, mde, alpha=0.05, power=0.8):
    """Users needed per group to detect an absolute lift of mde with a two-sided test."""
    p_new = p_base + mde
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    p_bar = (p_base + p_new) / 2
    numerator = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar))
                 + z_beta * np.sqrt(p_base * (1 - p_base) + p_new * (1 - p_new))) ** 2
    return int(np.ceil(numerator / mde ** 2))

n = sample_size_per_group(p_control, p_treatment - p_control)
n

In [ ]:
# Each user either converts (1) or does not (0)
control = rng.binomial(1, p_control, n)
treatment = rng.binomial(1, p_treatment, n)

print(f"Control:   {control.sum():,} conversions out of {n:,} users = {control.mean():.2%}")
print(f"Treatment: {treatment.sum():,} conversions out of {n:,} users = {treatment.mean():.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

rates = [control.mean(), treatment.mean()]
# Normal approximation for the 95% confidence interval of each rate
errors = [1.96 * np.sqrt(r * (1 - r) / n) for r in rates]

ax.bar(["Control (A)", "Treatment (B)"], rates, yerr=errors, capsize=8, color=["grey", "tab:blue"])
ax.set_ylabel("Conversion rate")
ax.set_title("Conversion rate by group (95% CI)")
plt.show()

### Two-Proportion Z-Test ###
The z-test uses the pooled proportion to measure how surprising the observed difference would be if there were no real difference.<br>
<br>
The confidence interval uses the unpooled standard errors, because it is an estimate of the true difference and not a test of the null.

In [ ]:
def two_proportion_ztest(x_a, x_b):
    """Two-sided z-test and 95% confidence interval for the difference in conversion rates."""
    n_a, n_b = len(x_a), len(x_b)
    p_a, p_b = x_a.mean(), x_b.mean()

    # Pooled proportion for the test statistic (assumes H0 is true)
    p_pool = (x_a.sum() + x_b.sum()) / (n_a + n_b)
    se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se_pool
    p_value = 2 * stats.norm.sf(abs(z))

    # Unpooled standard error for the confidence interval
    se_diff = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    margin = stats.norm.ppf(0.975) * se_diff
    ci = (p_b - p_a - margin, p_b - p_a + margin)

    return {"lift": p_b - p_a, "z": z, "p_value": p_value, "ci_95": ci}

result = two_proportion_ztest(control, treatment)
print(f"Observed lift:  {result['lift']:+.2%}")
print(f"z statistic:    {result['z']:.3f}")
print(f"p-value:        {result['p_value']:.4f}")
print(f"95% CI for lift: [{result['ci_95'][0]:+.2%}, {result['ci_95'][1]:+.2%}]")

In [ ]:
# Cross-check with a chi-square test on the 2x2 table.
# Without the continuity correction it gives the same p-value as the z-test.
table = np.array([
    [control.sum(), n - control.sum()],
    [treatment.sum(), n - treatment.sum()],
])
chi2, p_chi2, _, _ = stats.chi2_contingency(table, correction=False)

print(f"z-test p-value:     {result['p_value']:.6f}")
print(f"chi-square p-value: {p_chi2:.6f}")

Notice that the confidence interval is more informative than the p-value alone. It shows the range of lifts consistent with the data, so we can judge whether even the low end of the range would be worth shipping.

### Sample Size ###
Before running a test we need to know how many users to enroll. The formula for users per group is

$$
n = \frac{\left(z_{1-\alpha/2}\sqrt{2\bar{p}(1-\bar{p})} + z_{1-\beta}\sqrt{p_A(1-p_A) + p_B(1-p_B)}\right)^2}{(p_B - p_A)^2}
$$

where $\bar{p} = (p_A + p_B)/2$. Smaller effects need much larger samples, because the required sample size grows with $1/\text{MDE}^2$.

In [ ]:
for mde in [0.005, 0.01, 0.015, 0.02, 0.03]:
    print(f"A lift of {mde:.1%} needs {sample_size_per_group(p_control, mde):,} users per group")

### Power by Simulation ###
The formula gives us the sample size, but it is useful to check it by simulation. For each sample size we run many experiments and count the share that reach significance. That share is the power.<br>
<br>
The helper below works directly on conversion counts, so we can run thousands of experiments at once.

In [ ]:
def pvalues_from_counts(conv_a, conv_b, n_users):
    """Two-sided p-values for conversion counts. Works on arrays of experiments."""
    p_a = conv_a / n_users
    p_b = conv_b / n_users
    p_pool = (conv_a + conv_b) / (2 * n_users)
    se = np.sqrt(p_pool * (1 - p_pool) * 2 / n_users)
    z = (p_b - p_a) / se
    return 2 * stats.norm.sf(np.abs(z))

In [ ]:
sims = 2000
sizes = np.arange(1000, 12001, 1000)

powers = []
for size in sizes:
    conv_a = rng.binomial(size, p_control, sims)
    conv_b = rng.binomial(size, p_treatment, sims)
    p_values = pvalues_from_counts(conv_a, conv_b, size)
    powers.append(np.mean(p_values < 0.05))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, powers, marker="o")
ax.axhline(0.8, linestyle="--", color="grey", label="80% power")
ax.axvline(n, linestyle=":", color="tab:blue", label=f"Planned n = {n:,}")
ax.set_xlabel("Users per group")
ax.set_ylabel("Power")
ax.set_title("Simulated power for a 1.5 point lift")
ax.legend()
plt.show()

The curve crosses 80% near the planned sample size, which matches the formula. Simulation is also useful when the formula does not apply, such as with unequal group sizes, complicated metrics, or non-binary outcomes.

### A/A Tests ###
An A/A test gives both groups the same experience. Because there is no real difference, the test should only reach significance in about $\alpha$ = 5% of experiments. If it does not, the testing procedure is broken.

In [ ]:
sims = 10000

conv_a = rng.binomial(n, p_control, sims)
conv_b = rng.binomial(n, p_control, sims)
p_values = pvalues_from_counts(conv_a, conv_b, n)

print(f"Share of A/A tests with p < 0.05: {np.mean(p_values < 0.05):.1%}")

### The Peeking Problem ###
Many teams check results every day and stop as soon as the p-value dips below 0.05. This is a common mistake. Each look is another chance for a false positive.<br>
<br>
To show this, we run A/A tests with daily checks over 30 days. A test counts as a false positive if it is significant on **any** day.

In [ ]:
sims = 2000
days = 30
users_per_day = 300  # per group, per day

# Daily conversions for each group in each simulated experiment
daily_a = rng.binomial(users_per_day, p_control, size=(sims, days))
daily_b = rng.binomial(users_per_day, p_control, size=(sims, days))

# Results so far, using all users up to each day
cum_users = users_per_day * np.arange(1, days + 1)
cum_a = daily_a.cumsum(axis=1)
cum_b = daily_b.cumsum(axis=1)
daily_p = pvalues_from_counts(cum_a, cum_b, cum_users)

checked_once = np.mean(daily_p[:, -1] < 0.05)
checked_daily = np.mean((daily_p < 0.05).any(axis=1))

print(f"False positives when checking once at the end: {checked_once:.1%}")
print(f"False positives when checking every day:       {checked_daily:.1%}")

In [ ]:
# Share of experiments that have already declared a winner by each day
declared_by_day = np.mean((daily_p < 0.05).cumsum(axis=1) > 0, axis=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, days + 1), declared_by_day, marker="o")
ax.axhline(0.05, linestyle="--", color="grey", label="Nominal 5%")
ax.set_xlabel("Day")
ax.set_ylabel("Share of A/A tests declared significant")
ax.set_title("Peeking inflates false positives")
ax.legend()
plt.show()

**How to avoid this**
  - Decide the sample size in advance and only analyze once when it is reached
  - Use sequential testing methods built for repeated looks, such as group sequential designs or always-valid p-values
  - Use an experimentation platform that handles this for you

### Multiple Comparisons ###
Testing several metrics or variants at once raises the chance of a false positive. The same logic applies to checking many segments after the fact.

In [ ]:
n_metrics = 10

# Chance of at least one false positive across independent metrics, each tested at 5%
print(f"Chance of at least one false positive across {n_metrics} metrics: {1 - 0.95 ** n_metrics:.1%}")

# Bonferroni correction: divide alpha by the number of tests
print(f"Bonferroni threshold for each metric: {0.05 / n_metrics:.4f}")

## Summary ##
  - An A/B test compares a control and a treatment group that are assigned at random
  - Set the MDE, $\alpha$, and power before the test. These determine the sample size.
  - The two-proportion z-test gives a p-value and a confidence interval for the lift
  - Simulation helps check power and confirm that false positives are controlled
  - Peeking at results early and testing many metrics both inflate false positives

## Further Reading ##
  - *Trustworthy Online Controlled Experiments* by Ron Kohavi, Diane Tang, and Ya Xu
  - Evan Miller's sample size calculator: https://www.evanmiller.org/ab-testing/sample-size.html
  - Evan Miller, *How Not To Run an A/B Test*: https://www.evanmiller.org/how-not-to-run-an-ab-test.html